# Majority Voting Pipeline: LLM-Based Self-Consistency for Student Question Classification

Single LLM outputs can be inconsistent or unreliable. Furthermore, LLM self-reported confidence scores are not reliable and need to be calibrated, and confidence score thresholds need to be properly verified with domain experts. Therefore, to improve confidence and accuracy, we use **self-consistency decoding**: generating multiple independent outputs for the same input and selecting the most common classification through **majority voting**.
This notebook implements a **two-stage classification pipeline** that uses **self-consistency majority voting** with LLMs to robustly classify student questions in the Process Control and Dynamics course.

### Two-Stage Classification Pipeline

_In the report, I split up Stage 1 into 2 sub-stages of (a) Intent classification followed by (b) Topic classification on technical questions (i.e. intent_category == "Technical Question"). But in this notebook, I combined these 2 sub-stages together as 1 stage to reduce LLM calls in this implementation._

1. **Stage 1 - Intent & Topic Classification**: Categorizes student input as "Technical Question", "Prestructural Irrelevant", or "Follow-up Answer", and identifies relevant course topics from the provided topic list.
2. **Stage 2 - Self-Consistency SOLO Taxonomy**: Generates 5 parallel LLM predictions of the SOLO (Structural Object Learning Outcomes) taxonomy level and applies majority voting to determine the final SOLO label consensus. If no consensus is reached, it will be routed to domain expert.

## Setup

- **aiohttp**: For asynchronous HTTP requests to batch API calls efficiently
- **pandas**: For handling the input/output data
- **pydantic**: For structured validation of LLM outputs
- **xml**: For formatting the API request payload

In [ ]:
import os
import json
import re
import asyncio
import aiohttp
import pandas as pd
import xml.sax.saxutils as xml_escape
from typing import List, Literal
from pydantic import BaseModel, ValidationError, Field

In [ ]:
# must have environment variables set for API key and base URL
api_key = os.getenv("NALA_API_KEY")
base_url = os.getenv("NALA_BASE_URL")

url = f"{base_url}/api/llm/"

## Domain Knowledge: Course Topics and SOLO Taxonomy Rubric

The classifier needs domain-specific context:
- **Topics**: 6 core topics from the Process Control course (e.g., Introduction to Process Control, Laplace Transforms, etc.)
- **SOLO Taxonomy Levels**: A framework for classifying cognitive complexity:
  - **Unistructural**: Single fact or definition
  - **Multistructural**: Multiple concepts from same topic
  - **Relational**: Comparisons and integration across topics
  - **Extended Abstract**: Real-world application of concepts

In [ ]:
# Topics and Grading Rubric Definitions
TOPICS = """
1. Introduction to Process Control: Covers the importance of process control and process dynamics, elements of the control system, concepts on process and process variables, control strategies using a blending system example, feedforward and feedback control strategies, the use of block diagram and specific objectives of process control.
2. Theoretical Models of Chemical Processes: Covers the concept of process model and modeling principles, the three modeling approaches, developing dynamic models represented as ordinary differential equations using conservation laws, and analysing dynamic models in blending process, stirred tank heating process and liquid storage systems.
3. Laplace Transforms: Covers the concept of Laplace transform, deriving Laplace transform of typical functions, the Laplace transform properties, computing the Laplace transform and determining the solutions of differential equations by Laplace transform techniques.
4. Transfer Functions: Covers the concept of transfer functions, describing the development of transfer functions, applying specific steps to derive transfer functions, explaining the properties of transfer functions and how to linearize nonlinear processes.
5. Dynamic Behaviour of First Order and Second Order Processes: Covers the different types of standard process inputs, the response of first-order processes, the response of integrating processes and the response of second-order processes using a non-interacting tank example.
6. Dynamic Response Characteristics of More Complicated Processes: Covers stability of linear systems, the relationship between location of poles of transfer functions (TFs) and stability, plotting poles and zeros of TF in a complex plane, applying more complex TFs in process response, TFs in process with time delays and approximating complicated TFs with simpler low-order models.
"""

GRADING_RUBRIC = {
    "Prestructural Irrelevant": "Asks about technically irrelevant matters on exams, grades, deadlines, logistics, administration, greetings, study advice, or any topic beyond the scope of the provided course topics.",
    "Unistructural": "Asks about a fact or definition.",
    "Multistructural": "Asks about listing or describing multiple concepts of the same topic.",
    "Relational": "Asks about causes, compares, analyzes, or integrates concepts from different topics.",
    "Extended Abstract": "Asks about application of topic concepts to appropriate real-world process control and dynamics scenarios."
}

## Structured Outputs: Pydantic Models for Type Safety

We use Pydantic to enforce validation of LLM responses. This ensures:
- **Type safety**: JSON fields match expected types
- **Validation**: Required fields are present
- **Consistency**: Same structure for all API responses

Two models:
- `CategoryTopicOutput`: Stage 1 result (category + topics)
- `SoloTaxonomyOutput`: Stage 2 result (reasoning + solo_label)

In [ ]:
# Define Pydantic models for structured outputs
class CategoryTopicOutput(BaseModel):
    category: Literal["Technical Question", "Prestructural Irrelevant", "Follow-up Answer"]
    topics: List[str]

# Note: 'reasoning' is defined first to enforce Chain-of-Thought generation order
class SoloTaxonomyOutput(BaseModel):
    reasoning: str = Field(..., description="Step-by-step reasoning. MUST be generated first.")
    solo_label: Literal["Unistructural", "Multistructural", "Relational", "Extended Abstract"]


## Utility Functions: Conversation History and Response Parsing

Before classifying a question, we maintain context by including recent conversation history. This helps the LLM understand follow-up questions and multi-turn dialogues.

In [ ]:
# retrieve recent conversation history for context in stage 1
def get_convo_history(df: pd.DataFrame, idx: int, conversation_id: str, num_messages: int = 2) -> str:
    convo_rows = df.loc[(df['conversation_id'] == conversation_id) & (df.index < idx)]
    recent_context = convo_rows.tail(num_messages)
    
    convo_history = []
    for _, row in recent_context.iterrows():
        sender = "User" if row['sender'] == 'user' else "Assistant"
        convo_history.append(f"{sender}: {row['text']}")
        
    return "\n".join(convo_history)

def parse_llm_response(text: str, model_class: type[BaseModel]) -> BaseModel:
    """Parse LLM response and validate against Pydantic model."""
    try:
        json_match = re.search(r'\{.*\}|\[.*\]', text, re.DOTALL)
        json_data = json.loads(json_match.group()) if json_match else json.loads(text)
        return model_class(**json_data)
    except (json.JSONDecodeError, ValidationError) as e:
        raise ValueError(f"Parsing failed: {e}")

## Asynchronous API Communication

We communicate with the LLM via async HTTP to maximize throughput. The function:
1. Constructs an XML-formatted request with system prompt, user prompt, and hyperparameters
2. Sends to the endpoint with proper authentication
3. Implements retry logic for resilience (up to 5 attempts with exponential backoff)
4. Parses the response structure to extract the message content

The API uses GPT-5 with configurable `reasoning_effort` and `max_tokens` for different use cases.

In [ ]:
async def call_llm_api_async(
    session: aiohttp.ClientSession, 
    system_prompt: str, 
    user_prompt: str, 
    reasoning_effort: str = "high", 
    verbosity: str = "medium", 
    max_tokens: int = 500, 
    max_retries: int = 5
):
    """Asynchronous API call with retry logic for gpt-5 format."""
    escaped_system_prompt = xml_escape.escape(system_prompt)
    escaped_user_prompt = xml_escape.escape(user_prompt)

    # Updated XML structure based on the gpt-5 requirements
    xml_body = f"""
    <llm_request>
        <model>gpt-5</model>
        <system_prompt>{escaped_system_prompt}</system_prompt>
        <hyperparams>
            <reasoning_effort>{reasoning_effort}</reasoning_effort>
            <verbosity>{verbosity}</verbosity>
            <max_tokens>{max_tokens}</max_tokens>
        </hyperparams>
        <user_prompt>STUDENT INPUT: {escaped_user_prompt}</user_prompt>
    </llm_request>
    """

    headers = {
        "Content-Type": "application/xml",
        "Authorization": f"Bearer {api_key}"
    }

    retries = 0
    while retries < max_retries:
        try:
            async with session.post(url, data=xml_body, headers=headers) as response:
                response.raise_for_status()
                res_json = await response.json()
                
                # New parsing logic for the gpt-5 response format
                raw_data = res_json.get('raw', {})
                output_list = raw_data.get('output', [])
                
                # Find the block where type is 'message'
                message_block = next((item for item in output_list if item.get('type') == 'message'), None)

                if message_block and 'content' in message_block and len(message_block['content']) > 0:
                    return message_block['content'][0]['text']
                else:
                    raise ValueError("No message block found in response.")
                    
        except (aiohttp.ClientError, ValueError) as e:
            retries += 1
            await asyncio.sleep(1) # async sleep to avoid blocking the event loop
            if retries == max_retries:
                raise RuntimeError(f"API call failed after {max_retries} retries: {e}")

## Stage 1: Intent & Topic Classification

**Purpose**: Classify the student's input and identify relevant course topics.

**Input**: Student's question + recent conversation history  
**Output**: 
- `intent_category`: One of {Technical Question, Prestructural Irrelevant, Follow-up Answer}
- `topics`: List of relevant course topics (empty if not a technical question)

**Why Stage 1 exists**:
- Filters irrelevant inputs (administrative, off-topic questions)
- Provides topic context to Stage 2 for more accurate taxonomy classification
- Single LLM call per question (no self-consistency sampling needed here)

In [ ]:
## determine user intent and map question to topic(s) in the list
async def stage_1_intent_topic_classification(session: aiohttp.ClientSession, input_text: str, convo_history: str = "") -> CategoryTopicOutput:
    system_prompt = f"""
    You are an expert classifier for student input intent and topic classification. Here is the recent conversation history (if any):
    {convo_history}
    
    Your task is to: 
    1. Classify the student input's intent into EXACTLY ONE of these three categories:
    - Technical Question: A question about course content, concepts, or technical details, including follow-up questions seeking clarification or further explanation.
    - Prestructural Irrelevant: {GRADING_RUBRIC['Prestructural Irrelevant']}    
    - Follow-up Answer: The student is responding to the assistant's previous prompts or questions, typically engaging with or attempting to answer them.
    
    2. If the category is "Technical Question", identify the most relevant topics from the topic list provided below. Return an empty list for topics if the category is NOT "Technical Question".
    The topic summaries are tagged to each topic. Only output the MOST RELEVANT TOPIC NAMES as given in the list.
        
    Available Course Topic List:
    {TOPICS}
    
    RESPONSE FORMAT - MUST be valid JSON with EXACTLY these two keys:
    {{
        "category": "MUST BE ONE OF: 'Technical Question', 'Prestructural Irrelevant', or 'Follow-up Answer'",
        "topics": ["list of relevant topics or empty list"]
    }}
    
    EXAMPLES:
    Example 1 - Technical Question:
    {{"category": "Technical Question", "topics": ["Transfer Functions", "Laplace Transforms"]}}
    
    Example 2 - Prestructural Irrelevant:
    {{"category": "Prestructural Irrelevant", "topics": []}}
    
    Example 3 - Follow-up Answer:
    {{"category": "Follow-up Answer", "topics": []}}
    
    Respond with ONLY the JSON object. Do not add any other text.
    """
    # response = await call_llm_api_async(session, system_prompt, input_text, temperature=0.2, top_p=0.9)
    response = await call_llm_api_async(
        session, 
        system_prompt, 
        input_text, 
        reasoning_effort="medium", 
        verbosity="medium", 
        max_tokens=300
    )
    return parse_llm_response(response, CategoryTopicOutput)

## Stage 2: Self-Consistency Majority Voting for SOLO Taxonomy

**Purpose**: Robustly classify the cognitive level of the student's question using the SOLO taxonomy.

### Why Self-Consistency?
LLMs can produce inconsistent outputs. Instead of trusting a single prediction, we:
1. Generate **5 independent predictions** in parallel for the same question
2. Each prediction includes step-by-step **reasoning** (Chain-of-Thought)
3. Use **majority voting** to select the most common SOLO level
4. Return the reasoning from the winning prediction

### How Majority Voting Works
- If 3+ out of 5 samples agree → Accept that label as the final classification
- If no consensus → Return "Human Review Required"
- More samples = higher confidence but also higher API cost

### Process Flow
```
Question + Topics
    ↓
[5 parallel API calls with reasoning_effort="high"]
    ↓
SOLO Label 1, Label 2, Label 3, Label 4, Label 5
    ↓
[Majority vote: Which label appears most?]
    ↓
Final SOLO Label + Reasoning
```

In [ ]:
## call the LLM api to get the single solo taxonomy classification for each sample
async def _fetch_single_solo_sample(session: aiohttp.ClientSession, system_prompt: str, input_text: str) -> SoloTaxonomyOutput:
    response = await call_llm_api_async(
        session, 
        system_prompt, 
        input_text, 
        reasoning_effort="high", 
        verbosity="medium", 
        max_tokens=300
    )
    return parse_llm_response(response, SoloTaxonomyOutput)


## run through 5 parallel API calls to get multiple SOLO labels for self-consistency majority voting
async def stage_2_solo_taxonomy(session: aiohttp.ClientSession, input_text: str, topics: List[str], num_samples: int = 5) -> List[SoloTaxonomyOutput]:
    rubric_guidelines = "\n".join([f"{key}: {value}" for key, value in GRADING_RUBRIC.items() if key != "Prestructural Irrelevant"])
    
    system_prompt = f"""
    You are a SOLO taxonomy classifier for a student's question.
    You are given the identified relevant course topics for the student's question as well as the SOLO Taxonomy Guidelines.
    
    Your task is to classify the question to the appropriate SOLO Taxonomy level. 
    CRITICAL: You must think step-by-step. Output your brief concise "reasoning" FIRST, referencing the topics context and SOLO Taxonomy Guidelines, before outputting the final "solo_label".

    Relevant Topics: {', '.join(topics)}
    
    SOLO Taxonomy Guidelines:
    {rubric_guidelines}
    
    Example Output Format:
    {{
        "reasoning": "The student is asking to compare the properties of feedforward and feedback control. Because they are comparing and integrating concepts, this aligns with the Relational level.",
        "solo_label": "Relational"
    }}
    
    Respond with ONLY a valid JSON object containing "reasoning" and "solo_label".
    """
    
    # parallel and concurrent execution of 5 api calls
    tasks = [_fetch_single_solo_sample(session, system_prompt, input_text) for _ in range(num_samples)]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Filter out exceptions to keep only successful Pydantic objects
    valid_responses = [res for res in results if isinstance(res, SoloTaxonomyOutput)]
    
    if not valid_responses:
        raise RuntimeError("All parallel API calls failed for self-consistency sampling.")
        
    return valid_responses

In [ ]:
# majority voting logic to determine final SOLO label from multiple samples
def final_consensus_logic(responses: List[SoloTaxonomyOutput]):
    if not responses:
        return "Human Review Required", "No valid responses generated."
        
    labels = [response.solo_label for response in responses]
    label_counts = {}
    for label in labels:
        label_counts[label] = label_counts.get(label, 0) + 1
    
    max_count = max(label_counts.values())
    total_valid = len(responses)
    
    # Require either a strict majority, or at least 2 matching votes if samples dropped
    if max_count >= (total_valid // 2) + 1 or max_count >= 2:
        majority_label = [label for label, count in label_counts.items() if count == max_count][0]
        # Fetch the reasoning from the first response that gave the winning label
        winning_response = next(response for response in responses if response.solo_label == majority_label)
        return majority_label, winning_response.reasoning
    else:
        return "Human Review Required", f"No strong consensus reached among valid samples. Votes: {label_counts}"

## Full Pipeline

**For each row:**
1. ✅ **Skip assistant messages** (only classify student inputs)
2. ✅ **Retrieve conversation history** (last 2 messages for context)
3. ✅ **Run Stage 1** → Get intent category and topics
4. ✅ **Run Stage 2 (if Technical Question)** → Run 5 parallel LLM calls, get consensus
5. ✅ **Log all results** → Individual labels + final label + reasoning
6. ✅ **Error handling** → Gracefully catch failures without stopping the pipeline

**Architecture**: Sequential processing row-by-row, but parallel API calls within each row (Stage 2 SOLO Taxonomy Classification).

**Output**: DataFrame with original data + classification results saved to CSV

**Additional columns together with original data in the output CSV**:
- `intent_category`: Stage 1 result
- `topics_classified`: Related course topics
- `solo_label_1` through `solo_label_5`: Individual predictions from Stage 2
- `final_solo_label`: Majority-voted consensus
- `reasoning`: Explanation for the consensus label
- `processing_error`: Any errors encountered

In [ ]:
async def process_dataframe_sequentially(df: pd.DataFrame, output_filename: str = "majority_voting_pipeline_results.csv"):
    """
    Processes the dataframe row-by-row sequentially.
    Stage 2 sampling (the 5 API calls) is still done concurrently for speed.
    """
    # Initialize the new columns with None
    new_columns = [
        "intent_category", "topics_classified", 
        "solo_label_1", "solo_label_2", "solo_label_3", "solo_label_4", "solo_label_5", 
        "final_solo_label", "reasoning", "processing_error"
    ]
    for col in new_columns:
        if col not in df.columns:
            df[col] = None

    # Open a single session for all sequential calls
    async with aiohttp.ClientSession() as session:
        for idx, row in df.iterrows():
            # Optional: Skip assistant rows if your DataFrame contains both
            if row.get('sender') == 'assistant':
                continue
                
            print(f"--- Processing row {idx} ---")
            
            # Truncate user input to 75 characters for cleaner console output
            user_input = str(row['text'])
            display_text = (user_input[:75] + '...') if len(user_input) > 75 else user_input
            print(f"Input: \"{display_text}\"")
            
            try:
                # 1. Get History
                history = get_convo_history(df, idx, row['conversation_id'])
                
                # 2. Run Stage 1
                s1_result = await stage_1_intent_topic_classification(session, row['text'], history)
                df.at[idx, 'intent_category'] = s1_result.category
                df.at[idx, 'topics_classified'] = ", ".join(s1_result.topics)
                
                print(f"Stage 1 -> Intent: {s1_result.category} | Topics: {s1_result.topics}")
                
                # 3. Run Stage 2 if applicable
                if s1_result.category == "Technical Question" and s1_result.topics:
                    # This step runs the 5 samples in parallel for the current row
                    s2_results = await stage_2_solo_taxonomy(session, row['text'], s1_result.topics, num_samples=5)
                    
                    # Log the individual labels (padding with None if any API calls failed inside the gather)
                    individual_labels_list = []
                    for i in range(5):
                        if i < len(s2_results):
                            label = s2_results[i].solo_label
                            df.at[idx, f'solo_label_{i+1}'] = label
                            individual_labels_list.append(label)
                        else:
                            df.at[idx, f'solo_label_{i+1}'] = "Sample Failed"
                            individual_labels_list.append("Sample Failed")
                    
                    # Calculate and log the final consensus
                    final_label, reasoning = final_consensus_logic(s2_results)
                    df.at[idx, 'final_solo_label'] = final_label
                    df.at[idx, 'reasoning'] = reasoning
                    
                    print(f"Stage 2 -> Samples: {individual_labels_list}")
                    print(f"        -> Consensus: {final_label}\n")
                else:
                    print("Stage 2 -> Skipped (Not a Technical Question with Topics)\n")
                    
            except Exception as e:
                # Catch any unexpected errors (e.g., total API failure, missing columns) to prevent the loop from crashing
                df.at[idx, 'processing_error'] = str(e)
                print(f"  -> Error on row {idx}: {e}\n")

    # Export the final DataFrame
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"Processing complete! Results saved to {output_filename}")

In [ ]:
df = pd.read_csv("NALA_full_data_cleaned.csv")
await process_dataframe_sequentially(df)